<a href="https://colab.research.google.com/github/kgkoushikghosh/DeploydockerimagesinEKS/blob/main/Welcome_to_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import chroma, faiss

/tmp/ipykernel_10357/3215780722.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader


In [9]:
loader = PyMuPDFLoader("/content/Acuvate Software Solutions.pdf")

In [11]:
documents = loader.load()

In [12]:
chunks = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=50).create_documents([doc.page_content for doc in documents])

In [13]:
chunks

[Document(metadata={}, page_content='Acuvate Software Solutions & Services \nJob Title: Agentic AI Architect'),
 Document(metadata={}, page_content='Job Title: Agentic AI Architect \nLocation: Hyderabad, Telangana, India \nEmployment Type: Full-Time'),
 Document(metadata={}, page_content='Employment Type: Full-Time \n  \nRole Overview'),
 Document(metadata={}, page_content='As an Agentic AI Architect at Acuvate Software Solutions & Services, you will lead the'),
 Document(metadata={}, page_content='Software Solutions & Services, you will lead the architectural'),
 Document(metadata={}, page_content='vision, design, and implementation of enterprise-grade agentic AI solutions that drive digital'),
 Document(metadata={}, page_content='transformation across industries. You’ll collaborate with Product, Data Engineering, and Cloud'),
 Document(metadata={}, page_content='with Product, Data Engineering, and Cloud teams'),
 Document(metadata={}, page_content='to build autonomous AI agents, orch

In [22]:
from openai import AzureOpenAI
from langchain_community.embeddings import OpenAIEmbeddings


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.2 MB/s eta 0:00:00


You need to add your `azure_endpoint` and `azure_key` to Colab's secrets manager. Click the "🔑" icon in the left panel, then add `AZURE_ENDPOINT` and `AZURE_KEY` with their respective values. This ensures your credentials are not exposed directly in the notebook.

In [23]:
import os
from google.colab import userdata
from langchain_openai import AzureOpenAIEmbeddings

AZURE_ENDPOINT = userdata.get('OPENAI_ENDPOINT')
AZURE_KEY = userdata.get('OPENAI_API_KEY')

embeddings = AzureOpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1024,
    azure_deployment="text-embedding-3-small",
    azure_endpoint=AZURE_ENDPOINT,
    api_key=AZURE_KEY,
    openai_api_version="2023-05-15" # You might need to adjust this API version
)

In [24]:
import chromadb

vectordb = chroma.Chroma.from_documents(documents=chunks, embedding=embeddings)

In [27]:
result = vectordb.similarity_search_with_relevance_scores("What the key responsibilities in the document?")

In [28]:
result

[(Document(metadata={}, page_content='agents & AI-driven workflows. \n  \nKey Responsibilities \nArchitecture & System Design'),
  0.18918752630451685),
 (Document(metadata={}, page_content='technical & business stakeholders. \n• Leadership and mentoring experience.'),
  0.10873180560182782),
 (Document(metadata={}, page_content='systems. \nGovernance, Ethics & Performance'),
  0.10116981804200331),
 (Document(metadata={}, page_content='governance). \nModeling & Implementation'),
  0.08118858721886757)]

In [29]:
retriever = vectordb.as_retriever()


Now that we have a retriever, we can use it to fetch relevant documents based on a query.